# ___Data wrangling for the paper draft; with `FRED 4.0`___
--------------------------------------

In [1]:
!python --version

Python 3.14.4


The system cannot find the path specified.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [31]:
fred = pd.read_csv(r"../../data/chapter2/FRED/FRED4_Entire_Database_2026.csv", low_memory=False, header=0, skiprows=range(1, 7), encoding="latin1") # FRED 4.0
meta = pd.read_csv(r"../../data/chapter2/FRED/FRED3_Column_Definitions_2021.csv", usecols=["column_id", "name", "units"], index_col="column_id")

lookup = pd.read_csv(r"../../data/chapter2/plant_lookup.csv", low_memory=False, encoding="latin1", usecols=["genus", "apweb.family", # let's stick the the family info from APG website
                                                                                                         "order", "group"], index_col="genus").rename(mapper={"apweb.family": "family"}, axis=1) 

In [32]:
# including all the traits Luke advised

COLLABORATION_GRADIENT_TRAITS = [
    "F00679", # RD
    "F00727", # SRL
    # "F00104", # RCT
]

CONSERVATION_GRADIENT_TRAITS = [
    "F00709", # RTD
    # "F00261", # RN
]

CHOSEN_ROOT_TRAITS = COLLABORATION_GRADIENT_TRAITS + CONSERVATION_GRADIENT_TRAITS

CHOSEN_TRAITS_DICT = {
    "F00709":  "RTD",
    # "F00261":  "RN",
    "F00679":  "RD",
    "F00727":  "SRL",
    # "F00104":  "RCT",
}

PLANT_TAXONOMY_ACCEPTED_COLUMNS = [
    "F01286", # Genus of plant according to The Plant List
    "F01287", # Species epithet of plant according to The Plant List
    "F01289", # Family of plant according to The Plant List
    "F01290"  # Order of plant.
]

BINOMINAL_NAME = ["F01286", "F01287"]
BINOMINAL_NAME_DATA_SOURCE = ["F00018", "F00019"]
ROOT_ORDER = ["F00056"]
TAXONOMY_COLUMNS_TO_CROSSCHECK = ["F01289", "F01290", # from FRED 4.0
                                  "family", "order", "group" # from the lookup table
]

THICK_ROOTED_FAMILIES = ["Zamiaceae", "Cycadaceae", "Magnoliaceae"]

In [33]:
meta.loc[BINOMINAL_NAME + BINOMINAL_NAME_DATA_SOURCE + CHOSEN_ROOT_TRAITS + ROOT_ORDER, :]

,name,units
column_id,,
F01286,Plant taxonomy_Accepted genus_TPL,NaN
F01287,Plant Taxonomy_Accepted species_TPL,NaN
F00018,Plant taxonomy_Genus_Data Source,NaN
F00019,Plant taxonomy_Species_Data source,NaN
F00679,Root diameter,mm
F00727,Specific root length (SRL),m/g
F00709,Root tissue density (RTD),g/cm3
F00056,Root order,NaN


In [34]:
# records with missing binominal names aren't useful to us 
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].isna().mean() # dropna(subset=BINOMINAL_NAME)

F01286    0.290605
F01287    0.314019
F01289    0.289702
F01290    0.289586
F00679    0.836937
F00727    0.832876
F00709    0.882283
dtype: float64

In [35]:
# that many records with missing binominal names???
fred.loc[:, BINOMINAL_NAME + BINOMINAL_NAME_DATA_SOURCE].isna().sum()

F01286    22539
F01287    24355
F00018    22521
F00019    23283
dtype: int64

In [36]:
# thought we could use the data source's binominal names where FRED's binominal names are missing but looks like that won't help :/
# drop all the rows that do not have genus and species names

fred.dropna(subset=BINOMINAL_NAME, inplace=True)

### ___$1^{st}$ order roots___
__________________

In [37]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00056==1")

,F01286,F01287,F01289,F01290,F00679,F00727,F00709,F00056
0,Dicranopteris,linearis,Gleicheniaceae,Gleicheniales,NaN,NaN,NaN,1.0
3,Cunninghamia,lanceolata,Cupressaceae,Cupressales,NaN,NaN,NaN,1.0
6,Magnolia,baillonii,Magnoliaceae,Magnoliales,NaN,NaN,NaN,1.0
11,Acacia,auriculiformis,Fabaceae,Fabales,NaN,NaN,NaN,1.0
15,Polyspora,axillaris,Theaceae,Ericales,NaN,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...
70350,Pinus,strobus,Pinaceae,Pinales,0.421385,17.805402,0.337226,1.0
70354,Tsuga,canadensis,Pinaceae,Pinales,0.373780,30.873103,0.204034,1.0
70358,Sciadopitys,verticillata,Sciadopityaceae,Cupressales,0.620386,29.212460,0.125136,1.0
70363,Cephalotaxus,harringtonii,Cephalotaxaceae,Cupressales,0.626291,21.950489,0.125251,1.0


In [38]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00056==1").loc[:, BINOMINAL_NAME].drop_duplicates()

,F01286,F01287
0,Dicranopteris,linearis
3,Cunninghamia,lanceolata
6,Magnolia,baillonii
11,Acacia,auriculiformis
15,Polyspora,axillaris
...,...,...
70295,Encephalartos,gratus
70299,Zamia,lucayana
70303,Chamaecyparis,pisifera
70328,Ephedra,distachya


In [39]:
fred.loc[:, CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00056==1").isna().mean().rename(index=CHOSEN_TRAITS_DICT)

RD        0.266157
SRL       0.668451
RTD       0.726195
F00056    0.000000
dtype: float64

### ___RD $\le$ 2.0 mm___
-----------------------

In [40]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 2.000").dropna(subset=CHOSEN_ROOT_TRAITS)

,F01286,F01287,F01289,F01290,F00679,F00727,F00709
236,Larix,gmelinii,Pinaceae,Pinales,0.319412,32.211568,0.354286
237,Larix,gmelinii,Pinaceae,Pinales,0.411555,25.480815,0.294286
238,Larix,gmelinii,Pinaceae,Pinales,0.527621,12.980828,0.382857
239,Larix,gmelinii,Pinaceae,Pinales,0.846975,6.250067,0.334286
241,Larix,gmelinii,Pinaceae,Pinales,0.307450,34.615414,0.382857
...,...,...,...,...,...,...,...
71902,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.740062,8.685170,0.224091
71903,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.739066,8.855820,0.228155
71904,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.666566,9.584048,0.228379
71905,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.639820,10.362667,0.265331


In [41]:
# focusing on rows that have data for all the three traits
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 2.000").dropna(subset=CHOSEN_ROOT_TRAITS).loc[:, BINOMINAL_NAME].drop_duplicates()

,F01286,F01287
236,Larix,gmelinii
266,Fraxinus,mandshurica
382,Populus,trichocarpa
386,Populus,tremula
827,Cinnamomum,micranthum
...,...,...
71813,Sorocea,muriculata
71814,Stachyarrhena,acuminata
71815,Trymatococcus,amazonicus
71816,Zygia,inaequalis


In [42]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 2.000").dropna(subset=CHOSEN_ROOT_TRAITS).isna().mean().rename(index=CHOSEN_TRAITS_DICT)

F01286    0.0
F01287    0.0
F01289    0.0
F01290    0.0
RD        0.0
SRL       0.0
RTD       0.0
dtype: float64

In [43]:
# inspect the coverage of thick rooted families in this subset
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"(F00679 <= 2.000) and F01289.isin(@THICK_ROOTED_FAMILIES)").dropna(subset=CHOSEN_ROOT_TRAITS).drop_duplicates(subset=PLANT_TAXONOMY_ACCEPTED_COLUMNS)

,F01286,F01287,F01289,F01290,F00679,F00727,F00709
832,Magnolia,odora,Magnoliaceae,Magnoliales,0.500991,24.405710,0.206190
7385,Magnolia,carsonii,Magnoliaceae,Magnoliales,0.743000,15.130000,0.158000
8087,Liriodendron,tulipifera,Magnoliaceae,Magnoliales,0.852750,15.188097,0.129000
8090,Magnolia,acuminata,Magnoliaceae,Magnoliales,0.982750,12.382596,0.141000
8093,Magnolia,denudata,Magnoliaceae,Magnoliales,0.804500,20.901448,0.111000
8096,Magnolia,grandiflora,Magnoliaceae,Magnoliales,1.224000,9.091114,0.137000
8099,Magnolia,kobus,Magnoliaceae,Magnoliales,0.641500,27.324764,0.122000
8102,Magnolia,macrophylla,Magnoliaceae,Magnoliales,1.099000,12.895006,0.134000
8105,Magnolia,stellata,Magnoliaceae,Magnoliales,0.731500,31.165607,0.096000
8108,Magnolia,tripetala,Magnoliaceae,Magnoliales,0.829250,17.996477,0.135000


In [44]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"(F00679 <= 2.000) and F01289.isin(@THICK_ROOTED_FAMILIES)").dropna(subset=CHOSEN_ROOT_TRAITS).drop_duplicates(subset=PLANT_TAXONOMY_ACCEPTED_COLUMNS).\
        groupby("F01289").agg({"F01289": "count"})

,F01289
F01289,
Cycadaceae,2
Magnoliaceae,29
Zamiaceae,3


In [67]:
cutoff_2mm_taxa = fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"(F00679 <= 2.000)").dropna(subset=CHOSEN_ROOT_TRAITS).loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS].\
    drop_duplicates(subset=BINOMINAL_NAME).sort_values(["F01289", "F01290", "F01286", "F01287"]).reset_index(drop=True)
cutoff_2mm_taxa

,F01286,F01287,F01289,F01290
0,Strobilanthes,dimorphotricha,Acanthaceae,Lamiales
1,Actinidia,kolomikta,Actinidiaceae,Ericales
2,Saurauia,herthae,Actinidiaceae,Ericales
3,Liquidambar,formosana,Altingiaceae,Saxifragales
4,Liquidambar,gracilipes,Altingiaceae,Saxifragales
...,...,...,...,...
1435,Dioon,rzedowskii,Zamiaceae,Cycadales
1436,Encephalartos,gratus,Zamiaceae,Cycadales
1437,Zamia,lucayana,Zamiaceae,Cycadales
1438,Alpinia,japonica,Zingiberaceae,Zingiberales


In [ ]:
# WE HAVE SPECIES CLASSIFIED UNDER DIFFERENT HIGHER TAXA IN THIS SUBSET!!!!!

### ___RD $\le$ 1.00 mm___
------------------

In [48]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 1.000").dropna(subset=CHOSEN_ROOT_TRAITS)

,F01286,F01287,F01289,F01290,F00679,F00727,F00709
236,Larix,gmelinii,Pinaceae,Pinales,0.319412,32.211568,0.354286
237,Larix,gmelinii,Pinaceae,Pinales,0.411555,25.480815,0.294286
238,Larix,gmelinii,Pinaceae,Pinales,0.527621,12.980828,0.382857
239,Larix,gmelinii,Pinaceae,Pinales,0.846975,6.250067,0.334286
241,Larix,gmelinii,Pinaceae,Pinales,0.307450,34.615414,0.382857
...,...,...,...,...,...,...,...
71902,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.740062,8.685170,0.224091
71903,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.739066,8.855820,0.228155
71904,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.666566,9.584048,0.228379
71905,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.639820,10.362667,0.265331


In [49]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 1.000").dropna(subset=CHOSEN_ROOT_TRAITS).loc[:, BINOMINAL_NAME].drop_duplicates()

,F01286,F01287
236,Larix,gmelinii
266,Fraxinus,mandshurica
382,Populus,trichocarpa
386,Populus,tremula
827,Cinnamomum,micranthum
...,...,...
71813,Sorocea,muriculata
71814,Stachyarrhena,acuminata
71815,Trymatococcus,amazonicus
71816,Zygia,inaequalis


In [50]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"F00679 <= 1.000").dropna(subset=CHOSEN_ROOT_TRAITS).isna().mean().rename(index=CHOSEN_TRAITS_DICT)

F01286    0.0
F01287    0.0
F01289    0.0
F01290    0.0
RD        0.0
SRL       0.0
RTD       0.0
dtype: float64

In [51]:
# inspect the thick rooted family coverage
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"(F00679 <= 1.000) and F01289.isin(@THICK_ROOTED_FAMILIES)").dropna(subset=CHOSEN_ROOT_TRAITS).drop_duplicates(subset=PLANT_TAXONOMY_ACCEPTED_COLUMNS)

,F01286,F01287,F01289,F01290,F00679,F00727,F00709
832,Magnolia,odora,Magnoliaceae,Magnoliales,0.500991,24.405710,0.206190
7385,Magnolia,carsonii,Magnoliaceae,Magnoliales,0.743000,15.130000,0.158000
8087,Liriodendron,tulipifera,Magnoliaceae,Magnoliales,0.852750,15.188097,0.129000
8090,Magnolia,acuminata,Magnoliaceae,Magnoliales,0.982750,12.382596,0.141000
8093,Magnolia,denudata,Magnoliaceae,Magnoliales,0.804500,20.901448,0.111000
8099,Magnolia,kobus,Magnoliaceae,Magnoliales,0.641500,27.324764,0.122000
8105,Magnolia,stellata,Magnoliaceae,Magnoliales,0.731500,31.165607,0.096000
8108,Magnolia,tripetala,Magnoliaceae,Magnoliales,0.829250,17.996477,0.135000
8309,Magnolia,yuyuanensis,Magnoliaceae,Magnoliales,0.510000,35.800000,0.140000
9472,Magnolia,macclurei,Magnoliaceae,Magnoliales,0.555370,17.040000,0.240000


In [52]:
fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"(F00679 <= 1.000) and F01289.isin(@THICK_ROOTED_FAMILIES)").dropna(subset=CHOSEN_ROOT_TRAITS).drop_duplicates(subset=PLANT_TAXONOMY_ACCEPTED_COLUMNS).\
    groupby("F01289").agg({"F01289": "count"})

,F01289
F01289,
Cycadaceae,2
Magnoliaceae,28
Zamiaceae,3


In [68]:
cutoff_1mm_taxa = fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS].query(r"(F00679 <= 1.000)").dropna(subset=CHOSEN_ROOT_TRAITS).loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS].\
    drop_duplicates(subset=BINOMINAL_NAME).sort_values(["F01289", "F01290", "F01286", "F01287"]).reset_index(drop=True)
cutoff_1mm_taxa

,F01286,F01287,F01289,F01290
0,Strobilanthes,dimorphotricha,Acanthaceae,Lamiales
1,Actinidia,kolomikta,Actinidiaceae,Ericales
2,Saurauia,herthae,Actinidiaceae,Ericales
3,Liquidambar,formosana,Altingiaceae,Saxifragales
4,Liquidambar,gracilipes,Altingiaceae,Saxifragales
...,...,...,...,...
1296,Dioon,rzedowskii,Zamiaceae,Cycadales
1297,Encephalartos,gratus,Zamiaceae,Cycadales
1298,Zamia,lucayana,Zamiaceae,Cycadales
1299,Alpinia,japonica,Zingiberaceae,Zingiberales


In [ ]:
# SUSPECT THIS SUBSET ALSO HAS SPECIES CLASSIFIED UNDER MULTIPLE HIGHER RANKS!!!!!!!!

In [92]:
# see what species we are losing going from 2mm to 1mm cutoff

species_lost_2mm_to_1mm = np.setdiff1d(cutoff_2mm_taxa.loc[:, BINOMINAL_NAME].apply(' '.join, axis=1).unique(), 
                                        cutoff_1mm_taxa.loc[:, BINOMINAL_NAME].apply(' '.join, axis=1).unique())# .size
species_lost_2mm_to_1mm

array(['Aglaia squamulosa', 'Agrostis stolonifera', 'Alchemilla arvensis',
       'Alchemilla xanthochlora', 'Alopecurus geniculatus',
       'Alopecurus pratensis', 'Aniba citrifolia', 'Aniba williamsii',
       'Anthoxanthum odoratum', 'Arabidopsis arenosa',
       'Arabidopsis thaliana', 'Arabis sagittata',
       'Arrhenatherum elatius', 'Barbarea vulgaris',
       'Beilschmiedia brachythyrsa', 'Beilschmiedia percoriacea',
       'Beilschmiedia robusta', 'Bellis perennis', 'Berula erecta',
       'Brassica rapa', 'Briza media', 'Bromus hordeaceus',
       'Bromus sterilis', 'Calamagrostis canescens',
       'Campanula glomerata', 'Campanula patula',
       'Campanula persicifolia', 'Campanula rapunculoides',
       'Capsella bursa-pastoris', 'Cardamine pratensis', 'Carex leporina',
       'Carex muricata', 'Carex ornithopoda', 'Carex praecox',
       'Carex vulpina', 'Centaurium erythraea', 'Cerastium glomeratum',
       'Cerastium pumilum', 'Cerastium semidecandrum',
       'Clino

In [93]:
species_lost_2mm_to_1mm = pd.DataFrame({"F01286": [_.split()[0] for _ in species_lost_2mm_to_1mm], "F01287": [_.split()[1] for _ in species_lost_2mm_to_1mm]})
species_lost_2mm_to_1mm

,F01286,F01287
0,Aglaia,squamulosa
1,Agrostis,stolonifera
2,Alchemilla,arvensis
3,Alchemilla,xanthochlora
4,Alopecurus,geniculatus
...,...,...
134,Veronica,officinalis
135,Veronica,serpyllifolia
136,Veronica,spicata
137,Viola,arvensis


In [94]:
# full taxonomic profile of the 139 species
species_lost_2mm_to_1mm = pd.merge(left=species_lost_2mm_to_1mm, left_on=BINOMINAL_NAME, right=cutoff_2mm_taxa, right_on=BINOMINAL_NAME, how="left").sort_values(["F01289", "F01290"] + BINOMINAL_NAME).reset_index(drop=True)
species_lost_2mm_to_1mm

,F01286,F01287,F01289,F01290
0,Berula,erecta,Apiaceae,Apiales
1,Ruscus,aculeatus,Asparagaceae,Asparagales
2,Bellis,perennis,Asteraceae,Asterales
3,Erigeron,acris,Asteraceae,Asterales
4,Hypochaeris,radicata,Asteraceae,Asterales
...,...,...,...,...
134,Galium,pumilum,Rubiaceae,Gentianales
135,Galium,verum,Rubiaceae,Gentianales
136,Sherardia,arvensis,Rubiaceae,Gentianales
137,Saxifraga,tridactylites,Saxifragaceae,Saxifragales


In [96]:
species_lost_2mm_to_1mm.to_csv(r"../../data/chapter2/FRED/subsets/species_lost_2mm_to_1mm_FRED4.csv", index=False)

### ___Subetting by RD $\le$ 1.00 mm___
---------------------------------------------------------------------------------------

In [198]:
# final subset for the phylogenetics work;
# using traits - RD, SRL and RTD
# using RD <= 1.00 mm as the criteria

subset = fred.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS + CHOSEN_ROOT_TRAITS + ROOT_ORDER].query(r"F00679 <= 1.000").reset_index(drop=True)
subset

,F01286,F01287,F01289,F01290,F00679,F00727,F00709,F00056
0,Populus,tremuloides,Salicaceae,Malpighiales,0.220000,NaN,NaN,NaN
1,Acer,negundo,Sapindaceae,Sapindales,0.280000,NaN,NaN,NaN
2,Juglans,nigra,Juglandaceae,Fagales,0.300000,NaN,NaN,NaN
3,Quercus,rubra,Fagaceae,Fagales,0.230000,NaN,NaN,NaN
4,Carya,glabra,Juglandaceae,Fagales,0.220000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
9679,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.740062,8.685170,0.224091,NaN
9680,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.739066,8.855820,0.228155,NaN
9681,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.666566,9.584048,0.228379,NaN
9682,Handroanthus,ochraceus,Bignoniaceae,Lamiales,0.639820,10.362667,0.265331,NaN


In [199]:
# examine the root orders within this RD <= 1.00 mm subset
subset.F00056.value_counts(dropna=False)

F00056
NaN    5542
1.0    1908
2.0    1230
3.0     510
4.0     307
5.0     143
6.0      26
7.0       9
8.0       6
9.0       3
Name: count, dtype: int64

In [200]:
# drop records where root order is > 3 ???
# this is concerning but Luke advised against mixing the two approaches
subset.query(r"F00056 > 3")

,F01286,F01287,F01289,F01290,F00679,F00727,F00709,F00056
14,Acer,saccharum,Sapindaceae,Sapindales,0.220000,70.500000,NaN,4.0
15,Acer,saccharum,Sapindaceae,Sapindales,0.210000,83.100000,NaN,5.0
16,Acer,saccharum,Sapindaceae,Sapindales,0.200000,131.100000,NaN,6.0
17,Acer,saccharum,Sapindaceae,Sapindales,0.230000,80.200000,NaN,7.0
20,Fraxinus,americana,Oleaceae,Lamiales,0.210000,108.200000,NaN,4.0
...,...,...,...,...,...,...,...,...
7501,Vitis,vinifera,Vitaceae,Vitales,0.371517,0.480000,NaN,4.0
7502,Vitis,vinifera,Vitaceae,Vitales,0.590948,0.090000,NaN,5.0
7506,Vitis,vinifera,Vitaceae,Vitales,0.600550,0.290000,NaN,4.0
9001,Dacrydium,balansae,Podocarpaceae,Araucariales,0.847640,6.220949,0.526719,4.0


In [72]:
subset.loc[:, BINOMINAL_NAME].drop_duplicates() # that's a good number of species

,F01286,F01287
0,Populus,tremuloides
1,Acer,negundo
2,Juglans,nigra
3,Quercus,rubra
4,Carya,glabra
...,...,...
9591,Sorocea,muriculata
9592,Stachyarrhena,acuminata
9593,Trymatococcus,amazonicus
9594,Zygia,inaequalis


In [202]:
# species record with values for RD, SRL & RTD
subset.dropna(subset=CHOSEN_ROOT_TRAITS).drop_duplicates(subset=BINOMINAL_NAME)

,F01286,F01287,F01289,F01290,F00679,F00727,F00709,F00056
69,Larix,gmelinii,Pinaceae,Pinales,0.319412,32.211568,0.354286,1.0
92,Fraxinus,mandshurica,Oleaceae,Lamiales,0.288462,78.846169,0.231206,1.0
128,Populus,trichocarpa,Salicaceae,Malpighiales,0.142951,447.167648,0.169294,1.0
132,Populus,tremula,Salicaceae,Malpighiales,0.175513,330.424588,0.169294,1.0
358,Cinnamomum,micranthum,Lauraceae,Laurales,0.502558,24.150012,0.209434,1.0
...,...,...,...,...,...,...,...,...
9591,Sorocea,muriculata,Moraceae,Rosales,0.643000,8.540000,0.469000,NaN
9592,Stachyarrhena,acuminata,Rubiaceae,Gentianales,0.376000,20.890000,0.416000,NaN
9593,Trymatococcus,amazonicus,Moraceae,Rosales,0.581000,7.460000,0.531000,NaN
9594,Zygia,inaequalis,Fabaceae,Fabales,0.367000,26.160000,0.362000,NaN


In [203]:
# species records with values for both RD & SRL
subset.dropna(subset=COLLABORATION_GRADIENT_TRAITS).drop_duplicates(subset=BINOMINAL_NAME)

,F01286,F01287,F01289,F01290,F00679,F00727,F00709,F00056
12,Acer,saccharum,Sapindaceae,Sapindales,0.450000,11.500000,NaN,2.0
18,Fraxinus,americana,Oleaceae,Lamiales,0.670000,9.800000,NaN,2.0
23,Viola,pubescens,Violaceae,Malpighiales,0.640000,13.000000,NaN,2.0
27,Hydrophyllum,canadense,Boraginaceae,Boraginales,0.870000,16.200000,NaN,2.0
69,Larix,gmelinii,Pinaceae,Pinales,0.319412,32.211568,0.354286,1.0
...,...,...,...,...,...,...,...,...
9591,Sorocea,muriculata,Moraceae,Rosales,0.643000,8.540000,0.469000,NaN
9592,Stachyarrhena,acuminata,Rubiaceae,Gentianales,0.376000,20.890000,0.416000,NaN
9593,Trymatococcus,amazonicus,Moraceae,Rosales,0.581000,7.460000,0.531000,NaN
9594,Zygia,inaequalis,Fabaceae,Fabales,0.367000,26.160000,0.362000,NaN


In [204]:
# choosing to continue with records that have data for all the three traits; RD, SRL & RTD
subset.dropna(subset=CHOSEN_ROOT_TRAITS, inplace=True)
subset.isna().sum()

F01286       0
F01287       0
F01289       0
F01290       0
F00679       0
F00727       0
F00709       0
F00056    3764
dtype: int64

In [205]:
lookup.loc[subset.F01286.unique(), :]

KeyError: "['Austroblechnum', 'Cranfillia', 'Pectinopitys', 'Helesia', 'Tetrapilus', 'Heptapleurum', 'Veronia', 'Prasoxylon', 'Pseudodictamnus', 'Chrysojasminum', 'Hesperostipa', 'Rubroshorea', 'Wurfbainia', 'Andesanthus', 'Imbralyx', 'Alseodaphnopsis', 'Hymenopus', 'Leptobalanus', 'Eumachia'] not in index"

In [ ]:
# we need to manually cross check these genera

# Austroblechnu
# Cranfilli
# Pectinopitys
# Helesia
# Tetrapilus
# Heptapleurum
# Veronia
# Prasoxylon
# Pseudodictamnus
# Chrysojasminum
# Hesperostipa
# Rubroshorea
# Wurfbainia
# Andesanthus
# Imbralyx
# Alseodaphnopsis
# Hymenopus
# Leptobalanus
# Eumachia

In [206]:
# for the genera that the lookup table has taxonomic data for, do the crosschecking
lookup_merged = pd.merge(left=subset.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS].query(r"F01286.isin(@lookup.index)").drop_duplicates(),
                         left_on="F01286", right=lookup, right_index=True, how="left").reset_index(drop=True)
lookup_merged

,F01286,F01287,F01289,F01290,family,order,group
0,Larix,gmelinii,Pinaceae,Pinales,Pinaceae,Pinales,Gymnosperms
1,Fraxinus,mandshurica,Oleaceae,Lamiales,Oleaceae,Lamiales,Angiosperms
2,Populus,trichocarpa,Salicaceae,Malpighiales,Salicaceae,Malpighiales,Angiosperms
3,Populus,tremula,Salicaceae,Malpighiales,Salicaceae,Malpighiales,Angiosperms
4,Cinnamomum,micranthum,Lauraceae,Laurales,Lauraceae,Laurales,Angiosperms
...,...,...,...,...,...,...,...
1269,Sorocea,muriculata,Moraceae,Rosales,Moraceae,Rosales,Angiosperms
1270,Stachyarrhena,acuminata,Rubiaceae,Gentianales,Rubiaceae,Gentianales,Angiosperms
1271,Trymatococcus,amazonicus,Moraceae,Rosales,Moraceae,Rosales,Angiosperms
1272,Zygia,inaequalis,Fabaceae,Fabales,Fabaceae,Fabales,Angiosperms


In [207]:
# family level coflicts
lookup_merged.query(r"F01289!=family")

,F01286,F01287,F01289,F01290,family,order,group
32,Hedycarya,arborea,Minimiaceae,Laurales,Monimiaceae,Laurales,Angiosperms
112,Nyssa,sylvatica,Nyssaceae,Cornales,Cornaceae,Cornales,Angiosperms
367,Sambucus,williamsii,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
504,Viburnum,dentatum,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
531,Apodytes,dimidiata,Metteniusaceae,Metteniusales,Icacinaceae,Icacinales,Angiosperms
631,Viburnum,tinus,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
700,Galearia,maingayi,Fabaceae,Fabales,Pandaceae,Malpighiales,Angiosperms
706,Nyssa,aquatica,Nyssaceae,Cornales,Cornaceae,Cornales,Angiosperms
875,Viburnum,stipitatum,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
1004,Cephalotaxus,harringtonii,Cephalotaxaceae,Cupressales,Taxaceae,Pinales,Gymnosperms


In [208]:
lookup_merged.query(r"F01289!=family").drop_duplicates()

,F01286,F01287,F01289,F01290,family,order,group
32,Hedycarya,arborea,Minimiaceae,Laurales,Monimiaceae,Laurales,Angiosperms
112,Nyssa,sylvatica,Nyssaceae,Cornales,Cornaceae,Cornales,Angiosperms
367,Sambucus,williamsii,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
504,Viburnum,dentatum,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
531,Apodytes,dimidiata,Metteniusaceae,Metteniusales,Icacinaceae,Icacinales,Angiosperms
631,Viburnum,tinus,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
700,Galearia,maingayi,Fabaceae,Fabales,Pandaceae,Malpighiales,Angiosperms
706,Nyssa,aquatica,Nyssaceae,Cornales,Cornaceae,Cornales,Angiosperms
875,Viburnum,stipitatum,Viburnaceae,Dipsacales,Adoxaceae,Dipsacales,Angiosperms
1004,Cephalotaxus,harringtonii,Cephalotaxaceae,Cupressales,Taxaceae,Pinales,Gymnosperms


In [209]:
# spelling mistakes in FRED are indicated with and asterisk*

# taxa that FRED has correctly

# Nyssa sylvatica - stick to FRED  https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:271953-1#higher-classification
# Sambucus williamsii - stick to FRED  https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:149409-1#higher-classification
# Viburnum dentatum - stick to FRED  https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:326264-2#higher-classification
# Apodytes dimidiata - stick to FRED  https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:434212-1#higher-classification
# Viburnum tinus - stick to FRED  https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:326271-2#higher-classification
# Nyssa aquatica - stick to FRED https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:302695-2#higher-classification
# Viburnum stipitatum - stick to FRED https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:77175761-1#higher-classification

# taxa that need intervention

# Hedycarya arborea - *Monimiaceae, Laurales  https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:581882-1#higher-classification
# Galearia maingayi - Pandaceae, Malpighiales https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:349146-1#higher-classification
# Cephalotaxus harringtonii - Cephalotaxaceae, Pinales https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:261818-1#higher-classification
# Cryptocarya acutifolia - Lauraceae, Laurales https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:463781-1#higher-classification
# Barringtonia fusicarpa - Lecythidaceae, Ericales https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:104061-1#higher-classification
# Castanopsis indica - Fagaceae, Fagales https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:295397-1#higher-classification


In [210]:
# correct the above in the subset and remerge with the lookup table to incorporate the corrections

subset.loc[subset.query(r"F01286=='Hedycarya'").index, "F01289"] = "Monimiaceae"
subset.loc[subset.query(r"F01286=='Hedycarya'").index, "F01290"] = "Laurales"

subset.loc[subset.query(r"F01286=='Galearia'").index, "F01289"] = "Pandaceae"
subset.loc[subset.query(r"F01286=='Galearia'").index, "F01290"] = "Malpighiales"

subset.loc[subset.query(r"F01286=='Cephalotaxus'").index, "F01289"] = "Cephalotaxaceae"
subset.loc[subset.query(r"F01286=='Cephalotaxus'").index, "F01290"] = "Pinales"

subset.loc[subset.query(r"F01286=='Cryptocarya'").index, "F01289"] = "Lauraceae"
subset.loc[subset.query(r"F01286=='Cryptocarya'").index, "F01290"] = "Laurales"

subset.loc[subset.query(r"F01286=='Barringtonia'").index, "F01289"] = "Lecythidaceae"
subset.loc[subset.query(r"F01286=='Barringtonia'").index, "F01290"] = "Ericales"

subset.loc[subset.query(r"F01286=='Castanopsis'").index, "F01289"] = "Fagaceae"
subset.loc[subset.query(r"F01286=='Castanopsis'").index, "F01290"] = "Fagales"

In [211]:
# remerging after corrections, this is necessary since these corrections may have solved some of the order level conflicts
# so before reconciling the order level conflicts, do the remerger
lookup_merged = pd.merge(left=subset.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS].query(r"F01286.isin(@lookup.index)").drop_duplicates(),
                         left_on="F01286", right=lookup, right_index=True, how="left").reset_index(drop=True)

In [212]:
# order level conflicts

# with pd.option_context("display.max_rows", 100, "display.max_columns", 10):
#     print(lookup_merged.query(r"F01290!=order").drop_duplicates())

lookup_merged.query(r"F01290!=order").drop_duplicates()

# even though our FRED subset has heaps of order level conflicts with the lookup table, U.PhyloMaker only expects the following columns:
# species, genus, family, species.relative, genus.relative
# don't waste time reconciling the orders :)

,F01286,F01287,F01289,F01290,family,order,group
15,Asplenium,bulbiferum,Aspleniaceae,Polypodiales,Aspleniaceae,Eupolypod II,Pteridophytes
18,Lomaria,discolor,Blechnaceae,Polypodiales,Blechnaceae,Eupolypod II,Pteridophytes
19,Parablechnum,novae-zelandiae,Blechnaceae,Polypodiales,Blechnaceae,Eupolypod II,Pteridophytes
20,Parablechnum,procerum,Blechnaceae,Polypodiales,Blechnaceae,Eupolypod II,Pteridophytes
28,Dacrydium,cupressinum,Podocarpaceae,Araucariales,Podocarpaceae,Pinales,Gymnosperms
...,...,...,...,...,...,...,...
1014,Dacrydium,balansae,Podocarpaceae,Araucariales,Podocarpaceae,Pinales,Gymnosperms
1015,Nageia,nagi,Podocarpaceae,Araucariales,Podocarpaceae,Pinales,Gymnosperms
1019,Chamaecyparis,pisifera,Cupressaceae,Cupressales,Cupressaceae,Pinales,Gymnosperms
1020,Ephedra,distachya,Ephedraceae,Ephedrales,Ephedraceae,Gnetales,Gymnosperms


In [173]:
# now, examine the genera that are missing in the lookup table
subset.loc[:, PLANT_TAXONOMY_ACCEPTED_COLUMNS].query(r"not F01286.isin(@lookup.index)").drop_duplicates().sort_values(by=BINOMINAL_NAME)

,F01286,F01287,F01289,F01290
9264,Alseodaphnopsis,andersonii,Lauraceae,Laurales
8172,Andesanthus,lepidotus,Melastomataceae,Myrtales
483,Austroblechnum,lanceolatum,Blechnaceae,Polypodiales
5589,Chrysojasminum,fruticans,Oleaceae,Lamiales
485,Cranfillia,fluviatilis,Blechnaceae,Polypodiales
9493,Eumachia,astrellantha,Rubiaceae,Gentianales
9494,Eumachia,podocephala,Rubiaceae,Gentianales
952,Helesia,tetraptera,Styracaceae,Ericales
7803,Heptapleurum,heptaphyllum,Araliaceae,Apiales
1153,Heptapleurum,minutistellatum,Araliaceae,Apiales


In [ ]:
# Alseodaphnopsis - okay https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:77165678-1#higher-classification
# Andesanthus - okay https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:77205900-1#higher-classification
# Austroblechnum - synonym of Blechnum chambersii - Aspleniaceae, Polypodiales https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:53460-3#higher-classification
# Chrysojasminum - okay https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:77144221-1#higher-classification
# Cranfillia - synonym of Blechnum fluviatile - Aspleniaceae, Polypodiales https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:17058940-1#higher-classification
# Eumachia - okay https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:77154315-1#higher-classification
# Helesia tetraptera - correct the spelling error in FRED - Halesia tetraptera - synonym of Halesia carolina https://powo.science.kew.org/taxon/urn:lsid:ipni.org:names:319532-2#higher-classification
# 

In [218]:
# now, cross check if any genera are classified under multiple families
np.where(np.array([subset.query(r"F01286==@g").loc[:, "F01289"].unique().size for g in subset.F01286.unique()]) > 1)

(array([], dtype=int64),)